# Git Branch Workflow

This notebook explains how branches relate to each other, how to keep them synchronized, how to inspect work that has not yet been merged, and how to reduce merge conflicts.

## 1. The basic idea

A **branch** is an independent line of development. Branches do not contain separate copies of every file; they are labels that point to commits in the same Git history.

A common team workflow uses a shared integration branch named `development` and short-lived feature or fix branches:

```text
main
  |
  +-- development
        |
        +-- feature/add-search
        +-- feature/update-dashboard
        +-- fix/login-timeout
```

- `main` contains the stable, released version.
- `development` is the shared branch where approved work is integrated.
- `feature/...` branches contain one focused piece of new work.
- `fix/...` branches contain one focused correction.

Unless your team uses another model, start new work from an updated `development` branch and merge it back through a Pull Request (PR).

## 2. How branches diverge and reconnect

When you create a feature branch, both branch names initially point to the same commit. The history only diverges after a new commit is made on one of them:

```text
A --- B --- C                 development
           \
            D --- E           feature/add-search
```

Here, `feature/add-search` was created from commit `C`. Commits `D` and `E` belong only to the feature branch. Meanwhile, `development` may receive new commits from other merged PRs:

```text
A --- B --- C --- F --- G     development
           \
            D --- E           feature/add-search
```

The feature branch is now **behind** `development` with respect to commits `F` and `G`. Synchronizing the branch means bringing those changes into the feature branch before it is merged.

## 3. Local branches, remote branches, and `origin`

Git keeps your local work on your computer. GitHub hosts remote branches. `origin` is the conventional name for the remote repository you cloned from.

```text
Your computer                         GitHub
--------------                        --------------------------
development       <--> origin/development
feature/add-search <--> origin/feature/add-search
```

- `development` is your local branch.
- `origin/development` is your local record of the remote branch after the last `git fetch`.
- `git push` sends local commits to GitHub.
- `git fetch` downloads remote branch information without changing your files.
- `git pull` fetches and integrates changes into your current local branch.

Important: `origin/development` is not a live connection. Run `git fetch origin` before relying on it for comparisons.

## 4. Start new work from an updated shared branch

Before creating a branch, make sure you are starting from the latest shared code:

```bash
git switch development
git status
git pull --ff-only
git switch -c feature/add-search
```

`git pull --ff-only` updates your local `development` only when Git can move it forward without creating an unexpected merge commit. If it cannot, stop and inspect the situation.

Use a descriptive, general branch name:

```text
feature/add-search
feature/update-dashboard
fix/login-timeout
docs/branch-workflow
```

## 5. Work and publish your branch

Make small, focused commits. Before every commit, inspect what you are about to include:

```bash
git status
git diff
git add <file-or-directory>
git commit -m "Add search filters"
```

Push a new branch and create its upstream connection:

```bash
git push -u origin feature/add-search
```

After this first push, `git push` is normally enough. Publish early so that the branch is backed up, visible to teammates, and available for a PR.

## 6. See branches and their relationships

Refresh remote information, then list branches:

```bash
git fetch origin
git branch                 # Local branches
git branch -r              # Remote branches
git branch -a              # Local and remote branches
```

Show a compact graph with branch labels:

```bash
git log --oneline --decorate --graph --all
```

Show each local branch, its upstream branch, and whether it is ahead or behind:

```bash
git branch -vv
```

In VS Code, the built-in Source Control view and Git Graph can visualize the same branch relationships.

## 7. Find work that has not been merged

First fetch, so Git knows the current remote state:

```bash
git fetch origin
```

List branches that are **not merged into** `origin/development`:

```bash
git branch -r --no-merged origin/development
```

List branches that **are already merged into** `origin/development`:

```bash
git branch -r --merged origin/development
```

Inspect the commits and file changes a specific branch would add to `development`:

```bash
git log --oneline origin/development..origin/feature/add-search
git diff --stat origin/development...origin/feature/add-search
git diff origin/development...origin/feature/add-search
```

The three-dot comparison uses the common ancestor as the starting point. It answers: *What did this branch introduce since it split from development?*

## 8. Inspect another branch without merging it

You can review or test a teammate's branch locally without merging it into your own branch. Start with a clean working tree:

```bash
git status
git fetch origin
git switch --track origin/feature/update-dashboard
```

This creates a local branch that tracks `origin/feature/update-dashboard`. You can now inspect the files, run tests, and compare it with `development`.

When finished, return to your own branch:

```bash
git switch feature/add-search
```

Switching branches changes files in your working directory, but it does not change the remote branches.

## 9. Synchronize your branch with `development`

While you work, other PRs may be merged into `development`. Update your feature branch before requesting final review. There are two common approaches. Your team should agree on one.

### Option A: Merge the latest `development` into your branch

```bash
git switch feature/add-search
git fetch origin
git merge origin/development
```

This preserves the existing history and creates a merge commit when needed. It is straightforward and safe for shared branches.

### Option B: Rebase your branch onto the latest `development`

```bash
git switch feature/add-search
git fetch origin
git rebase origin/development
```

Rebase replays your branch commits on top of the newest `development`, producing a linear history. Do not rebase a branch that other people are actively building on unless the team has explicitly agreed to it. If you rebase a branch already pushed to GitHub, use `git push --force-with-lease`, never plain `--force`.

## 10. Avoid and resolve merge conflicts

A conflict occurs when Git cannot safely choose between overlapping changes. It is common when two branches edit the same lines, rename the same file differently, or make incompatible configuration changes.

### Prevent conflicts

- Start from an updated `development` branch.
- Keep each branch small, focused, and short-lived.
- Pull or fetch `development` regularly, especially before changing shared files.
- Coordinate before editing shared files such as dependency files, Docker configuration, environment templates, or central documentation.
- Make small commits with clear messages.
- Do not mix unrelated refactoring with a feature change.

### Resolve a conflict

If a merge or rebase stops because of a conflict:

```bash
git status
# Edit the files marked as conflicted and choose the correct final content
git add <resolved-file>
git merge --continue    # When resolving a merge
# or
git rebase --continue   # When resolving a rebase
```

Read both sides of a conflict before choosing a result. The correct answer is often a combination of both changes, not simply one side. Run relevant tests after resolving conflicts.

To cancel safely and return to the state before the operation:

```bash
git merge --abort
# or
git rebase --abort
```

## 11. Pull Request flow

A Pull Request is the team's review and integration point:

```text
Update development
      |
Create feature branch
      |
Commit and push focused work
      |
Open Pull Request: feature/add-search -> development
      |
Review, test, and synchronize with development
      |
Approve and merge
      |
Everyone updates their local development branch
```

Before merging, verify that the PR has the correct base branch, contains only the intended changes, has passed relevant tests, and has been synchronized with `development` if the team requires it.

## 12. After a Pull Request is merged

A merge on GitHub does not automatically update your computer. Refresh your local shared branch:

```bash
git switch development
git pull --ff-only
```

Then remove the merged feature branch when you no longer need it:

```bash
git branch -d feature/add-search
git push origin --delete feature/add-search
```

`git branch -d` refuses to delete a branch that is not merged into your current branch. That is a useful safety check.

## 13. Keep your working tree safe

Before switching branches, merging, rebasing, or pulling, check:

```bash
git status
```

Ideally Git reports `nothing to commit, working tree clean`. If you have unfinished changes, choose deliberately:

- Commit them if they are a coherent unit of work.
- Stash them temporarily if they are not ready to commit.
- Discard them only when you are certain they are unnecessary.

Example stash workflow:

```bash
git stash push -m "Work in progress: search filters"
git switch development
# Later
git switch feature/add-search
git stash pop
```

Never commit passwords, API keys, tokens, or local environment files. Use `.gitignore` for real secret files and commit a sanitized `.example` template instead.

## Quick reference

### Start a feature

```bash
git switch development
git pull --ff-only
git switch -c feature/my-change
```

### Publish work

```bash
git status
git diff
git add <files>
git commit -m "Describe the change"
git push -u origin feature/my-change
```

### Check unmerged remote branches

```bash
git fetch origin
git branch -r --no-merged origin/development
```

### Synchronize your feature branch

```bash
git switch feature/my-change
git fetch origin
git merge origin/development
```

## Golden rules

> Start new work from an updated shared branch.

> Keep branches focused and synchronize them before merging.

> Inspect the actual diff, not only commit messages, before approving a Pull Request.

> Check `git status` before any operation that changes branches or history.